In [1]:
import copy

import config as cfg
from data_acquirer import get_data, gen_data
from data_processor import process
from solver import solve_pdpcr,mass_solve, SolvedInstance
import visualization
from operator import itemgetter
import matplotlib.pyplot as plt

import pandas as pd
import time

In [2]:
"""
SOLVE_METHOD ={
1: solve_vrp_sumofcosts,
2: solve_flow_makespan,
3: solve_vrp_makespan,
4: solve_vrp_ver2
}
"""

method = 2
random_data = True
total_robots = 5
total_tasks = 50
data_name = 'data4'

In [ ]:

is_depot_task = False
if random_data:
    raw = gen_data(total_robots, total_tasks, False, is_depot_task)
else:
    raw = get_data(data_name,'csv', is_depot_task)

In [ ]:
try:
    raw.depot.coor = raw.depot.pickup_coor
except Exception as e:
    pass
visualization.plot_map(raw)

In [ ]:
processed_data = process(raw, method) 
solve_result = solve_pdpcr(processed_data, total_robots, method)

In [ ]:
solve_result['sol']

In [ ]:
sol = solve_result['sol']
x = solve_result['x']
f = solve_result['f']

In [ ]:
sol.get_value_df(f)

### Quick visualization

In [ ]:
df_x = sol.get_value_df(x,key_column_names=['from','to'])
df_x_moved = df_x[df_x['value']!=0]
np_moved = df_x_moved.to_numpy()
task_idx_mapper = {task.index: task for task in processed_data.tasks}
task_idx_mapper[0] =processed_data.depot

In [ ]:
solve_result['z']

In [ ]:
processed_data.tasks

In [ ]:
processed_data.depot

In [ ]:
depot = raw.depot
plt.scatter(depot.coor[0], depot.coor[1],c= 'C3', label = 'depot')
for moved_edge in np_moved:
    from_task_idx = int(moved_edge[0])
    to_task_idx = int(moved_edge[1])
    if from_task_idx ==0:
        frm_task_dest = task_idx_mapper[from_task_idx].coor
    else:
        frm_task_dest = task_idx_mapper[from_task_idx].dest_coor
    if to_task_idx ==0:
        to_task_pickup = task_idx_mapper[to_task_idx].coor
    else:
        to_task_pickup = task_idx_mapper[to_task_idx].t
    xs = [frm_task_dest[0],to_task_pickup[0]]
    ys = [frm_task_dest[1],to_task_pickup[1]]
    plt.plot(xs, ys)
for task in raw.tasks:
        pickup_coor = task.pickup_coor
        dest_coor = task.dest_coor
        plt.scatter(pickup_coor[0], pickup_coor[1],c = 'C0',)
        plt.scatter(dest_coor[0], dest_coor[1], c ='C1',)
        xs = [pickup_coor[0],dest_coor[0]]
        ys = [pickup_coor[1],dest_coor[1]]
        plt.plot(xs, ys, c = cfg.PATH_COLOR[task.n])

In [ ]:
processed_data.tasks

In [ ]:
MOVED_EDGE_GETTER ={
    1:visualization.get_moved_edges,
    2:visualization.get_moved_edges_flow
}

In [ ]:
task_idx_mapper = {task.index: task for task in processed_data.tasks}

In [ ]:
node_idx_mapper = {node.index: node for node in processed_data.tasks}
moved_edges = MOVED_EDGE_GETTER[method](solve_result)
paths = visualization.get_paths(moved_edges,total_robots)
tasks_path = visualization.get_tasks_path(paths, processed_data.nodes)
paths_coor = visualization.get_paths_coor(paths,node_idx_mapper)
edge_paths = visualization.get_edge_paths(paths)
if method ==2:
    node_idx_mapper = {node.index: node for node in processed_data.nodes[:-1]}

## Visualization

In [ ]:
import networkx as nx

In [ ]:
G = nx.DiGraph()
V =[node for node in node_idx_mapper.keys()]
E = moved_edges
pos = {idx: node.coor for idx,node in node_idx_mapper.items()}
        
G.add_nodes_from(V)        
G.add_edges_from(E)

color_map = []
for node in V:
    if node ==0:
        color_map.append('red')
    elif node_idx_mapper[node].node_type ==1:
        color_map.append('C0')
    elif node_idx_mapper[node].node_type ==2:
        color_map.append('C1')

In [ ]:
edges_color = []
edges_style = []
for edge in E:
    for i in range(len(edge_paths)):
        if edge in edge_paths[i]:
            edges_color.append(f'C{i}')
            edges_style.append((1,(3+i*2,5)))

In [ ]:
nx.draw(G,pos = pos, node_color = color_map,node_size =50, edge_color = edges_color, style = edges_style, arrowsize =15)
if random_data == False:
    plt.savefig(f"img/{data_name}result.png", format="PNG")
else:
    plt.savefig(f"img/{seed_num}_{current_time}result_random.png", format="PNG")
plt.show()

## Mass Solve

In [ ]:
Rs = [2,3,4]
Ts = [5,10,15]

Rs = [1,2] 
Ts = [5]
total_instances = 5
cfg.LOG_OUTPUT = False
full_solved_instances = []
for T in Ts:
    for R in Rs:
        print(f'__________________________solving for {T} tasks and {R} robot__________________________')
        solved_instances = mass_solve(T,R,2,total_instances = total_instances,log_output = False)
        [full_solved_instances.append(solved_instance) for solved_instance in solved_instances]

## Mass solve compare

In [ ]:
Rs = [1,2,3,4]
Ts = [5,10,15]

# Rs = [2,]
# Ts = [5]
total_instances = 4
cfg.LOG_OUTPUT = False
full_solved_instances_vrp = []
full_solved_instances_flow = []
for T in Ts:
    print(f"number of tasks {T}")
    for R in Rs:
        print(f"number of robots {R}")
        for instance in range(total_instances):
            print(f"current instance {instance}")
            raw = gen_data(R, T, False, False)
            processed_data_flow = process(raw,2)
            tik = time.time()
            solved_result_flow = solve_pdpcr(processed_data_flow, R, 2)
            tok = time.time()
            solved_time = tok - tik
            solved_flow = SolvedInstance(1,processed_data_flow,R,solved_result_flow,solved_time,instance)
            full_solved_instances_flow.append(solved_flow)
            
            processed_data_vrp = process(raw,5)
            tik = time.time()
            solved_result_vrp = solve_pdpcr(processed_data_vrp, R, 5)
            tok = time.time()
            solved_time = tok - tik
            solved_vrp = SolvedInstance(1,processed_data_vrp,R,solved_result_vrp,solved_time,instance)
            full_solved_instances_vrp.append(solved_vrp)

In [ ]:
# full_solved_instances_vrp[0].solve_result['sol'].objective_value - full_solved_instances_flow[0].solve_result['sol'].objective_value
for i in range(len(full_solved_instances_flow)):
    print(f'i = {i}')
    flow_sol = full_solved_instances_flow[i].solve_result['sol']
    vrp_sol = full_solved_instances_vrp[i].solve_result['sol']
    try:
        flow_obj = flow_sol.objective_value
        vrp_obj = vrp_sol.objective_value
    except Exception as e:
        print(e)
        pass
    print(flow_obj -vrp_obj)

In [ ]:
def results_to_df(full_solved_instances, compact = True):
    solved_dict = [solve_instance.__dict__ for solve_instance in full_solved_instances]
    df_solved = pd.DataFrame(solved_dict)
    if compact == True:
        df_solved.drop(['data','seed','instance_num'],axis =1, inplace = True)
    return df_solved

In [ ]:
df_results = results_to_df(full_solved_instances_vrp)
df_results = df_results.round(2)

In [ ]:
df_results

In [ ]:
total_tasks = [len(solved_instance.data.tasks) for solved_instance in full_solved_instances]
total_robots = [(solved_instance.total_robots) for solved_instance in full_solved_instances]
# total_nodes = [len(solved_instance.data.nodes) for solved_instance in full_solved_instances]
orig_tasks = []
for T in Ts:
    for R in Rs:
        for _ in range(total_instances):
            orig_tasks.append(T)

In [ ]:
df_results['total_tasks'] = total_tasks
# df_results['total_nodes'] = total_nodes
df_results['orig_tasks'] = orig_tasks

In [ ]:
df_results = df_results[['total_robots','orig_tasks','solve_time','gap']]

In [ ]:
df_results

In [ ]:
print(df_results.groupby(['total_robots','orig_tasks']).mean().reset_index().round(2).to_latex())

In [ ]:
df_final = df_results.groupby(['total_robots','orig_tasks']).mean().reset_index().round(2)
df_final

## Compare big M

In [ ]:
import copy

import config as cfg
from data_acquirer import get_data, gen_data
from data_processor import process
from solver import solve_pdpcr,mass_solve, SolvedInstance
import visualization
from operator import itemgetter
import matplotlib.pyplot as plt

import pandas as pd
import time

In [ ]:
method = 2
random_data = True
total_robots = 3
total_tasks = 25

cfg.RELAXED = False
cfg.USE_INDICATOR = False
cfg.BENDER_STRAT = 0

In [ ]:
big_ms = [10,10**2,10**3,10**4, 10**5, 10**6,10**7]
big_ms = [10**5]
methods = [2]

In [ ]:
raw = gen_data(total_robots, total_tasks, False, False)
solve_results = []
for big_m in big_ms:
    cfg.BIG_M = big_m
    for method in methods:
        processed_data = process(raw, method)
        solve_result = solve_pdpcr(processed_data, total_robots, method)
        solve_results.append(solve_result)

In [ ]:
[solve_result['sol'].objective_value for solve_result in solve_results if solve_result['sol'] is not None]

## Visualize old

In [ ]:
import networkx as nx

In [ ]:
G = nx.DiGraph()
V =[node for node in node_idx_mapper.keys()]
E = moved_edges
pos = {idx: node.coor for idx,node in node_idx_mapper.items()}
        
G.add_nodes_from(V)        
G.add_edges_from(E)

color_map = []
for node in V:
    if node ==0:
        color_map.append('red')
    elif node_idx_mapper[node].node_type ==1:
        color_map.append('C0')
    elif node_idx_mapper[node].node_type ==2:
        color_map.append('C1')

In [ ]:
rnd_list = {1:20,2:30}
max(rnd_list.values())

In [ ]:
rnd_list.values()